In [ ]:
import re, numpy as np, time, matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

## Load Data

In [140]:
def read_fasta(path):
    headers, sequences = [], []
    h, buf = None, []
    
    with open(path, 'r') as f:
        for line in f:
            if not line: 
                continue
            if line[0] == '>':
                if h is not None:
                    sequences.append(''.join(buf).upper())
                    buf = []
                h = line[1:].strip()
                headers.append(h)
            else:
                buf.append(line.strip())
        if h is not None:
            sequences.append(''.join(buf).upper())
            
    return headers, sequences

def load_labels(path):
    label_dict = {}
    with open(path, 'r') as f:
        for line in f:
            if not line:
                continue
            parts = line.strip().split('\t')
            header = parts[0][1:]
            if parts[1] == "TRUE":
                label = 1.0
            else:
                label = 0.0
            label_dict[header] = label
    return label_dict

## Data Processing

In [123]:
# mapping ACGT to 0123
# mapping acgt to 0123
# mapping N and others to 4

ENCODE = np.full(256, 4, dtype=np.int64)
for ch, idx in zip(b"ACGTacgt", [0, 1, 2, 3, 0, 1, 2, 3]):
    ENCODE[ch] = idx

# reverse complement ACGTN -> TGCAN -> 32104
REV_COMP = torch.tensor([3, 2, 1, 0, 4]) 

In [168]:
class SeqDataset(Dataset):
    def __init__(self, headers, sequences, labels):
        self.headers = headers
        self.sequences = sequences
        self.labels = np.asarray(labels, dtype=np.float32)

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.headers[idx], self.sequences[idx].encode('ascii', 'ignore'), self.labels[idx]

In [ ]:
def collate_pad(batch):
    headers, seq_bytes, labels = zip(*batch)
    idx_list = [ENCODE[np.frombuffer(sb, dtype=np.uint8)] for sb in seq_bytes]
    max_length = max(x.size for x in idx_list)
    batch_size = len(idx_list)
    
    X = torch.zeros(batch_size, 5, max_length, dtype=torch.float32)   # channels: A, C, G, T, N
    mask = torch.zeros(batch_size, max_length, dtype=torch.bool)      # padding mask
    
    for i, idx in enumerate(idx_list):
        length = idx.size
        X[i, idx, torch.arange(length)] = 1.0
        mask[i, :length] = (idx != 4)   # invalid positions: N or everything beyond the length
    
    Y = torch.tensor(labels, dtype=torch.float32)
    return headers, X, mask, Y

## CNN Network

In [170]:
# Reverse complement convolution layer
# input: one-hot (B, 5, L)
# output: RC-invariant feature map (B, C, L)

def rc_kernel(k: torch.Tensor):
    # Reverse complement the convolution kernel
    # k: (out_channels, in_channels, length)
    k_flip = k.flip(-1)
    return k_flip.index_select(1, REV_COMP.to(k.device))

class RCFirstConv1d(nn.Module):
    
    def __init__(self, out_channels, kernel_size=15, dilation=1, bias=True, dropout=0.1):
        super().__init__()
        assert kernel_size % 2 == 1
        pad = (kernel_size // 2) * dilation
        self.conv = nn.Conv1d(5, out_channels, kernel_size, padding=pad, dilation=dilation, bias=bias)
        self.bn   = nn.BatchNorm1d(out_channels)
        self.do   = nn.Dropout(dropout)

    def forward(self, x):
        
        y1 = self.conv(x)                    # (B, C, L)
        weight = self.conv.weight            # (C_out, 5, K)
        b = self.conv.bias
        
        weight_rc = rc_kernel(weight)        # (C_out, 5, K)
        y2 = F.conv1d(x, weight_rc, None, stride=1, padding=self.conv.padding[0], dilation=self.conv.dilation[0])
        y2 = y2.flip(-1)
        if b is not None:
            y1 = y1 + b.view(1, -1, 1)
            y2 = y2 + b.view(1, -1, 1)

        y = 0.5 * (y1 + y2)                  # RC-invariant feature map
        y = self.dropout(F.gelu(self.bn(y)))
        return y

In [171]:
class ConvBlock(nn.Module):
    def __init__(self, c_in, c_out, kernel_size=9, dilation=1, dropout=0.1):
        super().__init__()
        pad = (kernel_size // 2) * dilation
        self.conv = nn.Conv1d(c_in, c_out, kernel_size, padding=pad, dilation=dilation)
        self.bn   = nn.BatchNorm1d(c_out)
        self.do   = nn.Dropout(dropout)
    
    def forward(self, x):
        return self.do(F.gelu(self.bn(self.conv(x))))

In [172]:
class RCInputInvariantCNN(nn.Module):
    def __init__(self, width=128, blocks=4, dilate=True):
        super().__init__()
        self.first = RCFirstConv1d(width, kernel_size=15, dilation=1, dropout=0.1)
        trunk, c = [], width
        for i in range(blocks):
            dilation = 2 ** i if dilate else 1
            trunk += [ConvBlock(c, width, kernel_size=9, dilation=dilation), ConvBlock(width, width, kernel_size=3, dilation=1)]
        self.trunk = nn.Sequential(*trunk)
        self.head  = nn.Sequential(nn.Linear(width, 128), nn.GELU(), nn.Dropout(0.2), nn.Linear(128, 1))

    @staticmethod
    def masked_avg_pool(z, mask):
        if mask is None: return z.mean(-1)
        m = mask.unsqueeze(1).float()
        return (z * m).sum(-1) / m.sum(-1).clamp_min(1.0)

    def forward(self, x, mask=None):
        z = self.first(x)
        z = self.trunk(z)
        z = self.masked_avg_pool(z, mask)             # (B, width)
        return self.head(z).squeeze(1)                # (B,)

## Train & Eval

In [173]:
headers, sequences = read_fasta(fasta_path)
label_dict = load_labels(label_path)
labels = [label_dict[h] for h in headers]

idx_tr, idx_te = train_test_split(
    np.arange(len(sequences)), test_size=0.2, stratify=labels, random_state=42
)
ds_tr = SeqDataset([headers[i] for i in idx_tr], [sequences[i] for i in idx_tr], [labels[i] for i in idx_tr])
ds_te = SeqDataset([headers[i] for i in idx_te], [sequences[i] for i in idx_te], [labels[i] for i in idx_te])

loader_tr = DataLoader(ds_tr, batch_size=8, shuffle=True, num_workers=0, collate_fn=collate_pad)
loader_te = DataLoader(ds_te, batch_size=8, shuffle=False, num_workers=0, collate_fn=collate_pad)

model = RCInputInvariantCNN(width=128, blocks=4, dilate=True).to('mps')
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = nn.BCEWithLogitsLoss()

In [174]:
for a, b, c in loader_tr:
    print(a)
    print(b.shape)
    print(c.shape)
    break

TypeError: can't assign a numpy.ndarray to a torch.BoolTensor

In [166]:
for ep in range(1, 5 + 1):
    model.train()
    running = 0.0
    for _, X, mask, Y in loader_tr:
        X, mask, Y = X.to('mps'), mask.to('mps'), Y.to('mps')
        logits = model(X, mask)
        loss = loss_fn(logits, Y)
        opt.zero_grad(); loss.backward(); opt.step()
        running += loss.item() * X.size(0)
    print(f"Epoch {ep}: train loss {running/len(ds_tr):.4f}")

ValueError: not enough values to unpack (expected 3, got 2)

In [ ]:
def run_train(fasta_path, label_path, batch_size=8, epochs=5, lr=1e-3, width=128, blocks=4, device='mps'):
    
    headers, sequences = read_fasta(fasta_path)
    label_dict = load_labels(label_path)
    labels = [label_dict[h] for h in headers]

    idx_tr, idx_te = train_test_split(
        np.arange(len(sequences)), test_size=0.2, stratify=labels, random_state=42
    )
    ds_tr = SeqDataset([headers[i] for i in idx_tr], [sequences[i] for i in idx_tr], [labels[i] for i in idx_tr])
    ds_te = SeqDataset([headers[i] for i in idx_te], [sequences[i] for i in idx_te], [labels[i] for i in idx_te])

    loader_tr = DataLoader(ds_tr, batch_size=batch_size, shuffle=True, num_workers=0, collate_fn=collate_pad)
    loader_te = DataLoader(ds_te, batch_size=batch_size, shuffle=False, num_workers=0, collate_fn=collate_pad)

    model = RCInputInvariantCNN(width=width, blocks=blocks, dilate=True).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    loss_fn = nn.BCEWithLogitsLoss()

    for ep in range(1, epochs + 1):
        model.train()
        running = 0.0
        for _, X, mask, Y in loader_tr:
            X, mask, Y = X.to(device), mask.to(device), Y.to(device)
            logits = model(X, mask)
            loss = loss_fn(logits, Y)
            opt.zero_grad(); loss.backward(); opt.step()
            running += loss.item() * X.size(0)
        print(f"Epoch {ep}: train loss {running/len(ds_tr):.4f}")

        # quick val
        model.eval()
        Ps, Ys = [], []
        with torch.no_grad():
            for _, X, mask, Y in loader_te:
                X, mask, Y = X.to(device), mask.to(device), Y.to(device)
                p = torch.sigmoid(model(X, mask)).detach().cpu().numpy()
                Ps.append(p); Ys.append(Y.detach().cpu().numpy())
        Ps = np.concatenate(Ps); Ys = np.concatenate(Ys)
        try:
            auroc = roc_auc_score(Ys, Ps)
            auprc = average_precision_score(Ys, Ps)
        except ValueError:
            auroc = float('nan'); auprc = float('nan')
        print(f"  Val AUROC {auroc:.4f} | AUPRC {auprc:.4f}")

In [161]:
fasta_path = '../data/vgp/all_vgp_tes.fa'
label_path = '../data/vgp/features.txt'
run_train(fasta_path, label_path)

ValueError: not enough values to unpack (expected 3, got 2)